# Unified Model Training Notebook

This notebook trains all recommendation models available in the project: LightGCN, Matrix Factorization, NCF, Node2Vec, and Cleora.
It uses the shared data loaders and preprocessing steps.

In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys

# Add project root to path if necessary
sys.path.append(os.path.abspath('..'))

from helpers.data_loaders import (
    load_movielens_data, 
    load_steam_data, 
    transform_movielenses_to_graph, 
    transform_steam_to_graph, 
    merge_graphs, 
    create_pyg_graph
)
from src.models.lightgcn import LightGCNGenerator
from src.models.matrix_factorization import MatrixFactorizationGenerator
from src.models.ncf import NCFGenerator
from src.models.node2vec import Node2VecGenerator
from src.models.cleora import CleoraGenerator
from src.evaluation import random_split, calculate_metrics, print_metrics

## 1. Load and Preprocess Data

In [3]:
print("Loading data...")
movies_df, ratings_df = load_movielens_data("datasets/movies/movies.csv", "datasets/movies/ratings.csv")
reviews_df_steam, items_df_steam = load_steam_data("datasets/steam/formatted_user_reviews.json", "datasets/steam/formatted_steam_games.json")

# Basic Interactions DataFrame
ratings_df_filtered = ratings_df[ratings_df['rating'] >= 4.0].copy()

movielens_interactions = pd.DataFrame({
    'user_id': 'movielens_user_' + ratings_df_filtered['userId'].astype(str),
    'item_id': 'movielens_item_' + ratings_df_filtered['movieId'].astype(str),
    'rating': ratings_df_filtered['rating']
})

steam_interactions = pd.DataFrame({
    'user_id': 'steam_user_' + reviews_df_steam['user_id'].astype(str),
    'item_id': 'steam_item_' + reviews_df_steam['app_id'].astype(str),
    'rating': 1.0 # Implicit
})

all_interactions = pd.concat([movielens_interactions, steam_interactions]).drop_duplicates(subset=['user_id', 'item_id'])
print(f"Total unique interactions: {len(all_interactions)}")

# Split Data
train_interactions, test_interactions = random_split(all_interactions, test_size=0.2)
print(f"Training: {len(train_interactions)}, Test: {len(test_interactions)}")

Loading data...
Total unique interactions: 12497401
Training: 9990389, Test: 2508908


## 2. Train LightGCN

In [ ]:
print("Training LightGCN...")
lightgcn = LightGCNGenerator(epochs=5, batch_size=4096)
lightgcn.train(train_interactions)
lightgcn.save('models/lightgcn_model.pth')
print("LightGCN saved.")

Training LightGCN...


Epoch 1/5:   0%|          | 0/2440 [00:00<?, ?it/s]

## 3. Train Matrix Factorization

In [ ]:
print("Training Matrix Factorization...")
mf = MatrixFactorizationGenerator(epochs=5, batch_size=4096)
mf.train(train_interactions, rating_col='rating')
mf.save('models/matrix_factorization_model.pth')
print("Matrix Factorization saved.")

## 4. Train NCF

In [ ]:
print("Training NCF...")
ncf = NCFGenerator(epochs=3, batch_size=1024)
ncf.train(train_interactions)
ncf.save('models/ncf_model.pth')
print("NCF saved.")

## 5. Train Node2Vec
Node2Vec uses a graph structure including genres for better embeddings.

In [ ]:
print("Preparing Graph for Node2Vec...")
# Construct generic graph
movielens_graph = transform_movielenses_to_graph(ratings_df.copy(), movies_df.copy())
steam_graph = transform_steam_to_graph(reviews_df_steam, items_df_steam)
merged_graph = merge_graphs(movielens_graph, steam_graph)

# Create PyG graph from training data only to avoid leakage
# Note: properly splitting a graph is complex. Here we use the training interactions 
# but keep the auxiliary genre edges.
train_edges = list(zip(train_interactions['user_id'], train_interactions['item_id']))
train_graph = {
    'user_nodes': merged_graph['user_nodes'],
    'item_nodes': merged_graph['item_nodes'],
    'genre_nodes': merged_graph['genre_nodes'],
    'user_item_edges': train_edges,
    'item_genre_edges': merged_graph['item_genre_edges']
}

G_pyg_train, node_to_int_id, int_id_to_node = create_pyg_graph(train_graph)

print("Training Node2Vec...")
node2vec = Node2VecGenerator(epochs=5, batch_size=128)
# Pass constructed graph
node2vec.train(train_interactions, edge_index=G_pyg_train.edge_index, num_nodes=G_pyg_train.num_nodes, node_map=node_to_int_id)
node2vec.save('models/node2vec_model.pth')
print("Node2Vec saved.")

## 6. Train Cleora
Cleora also benefits from genre connections.

In [ ]:
print("Preparing Data for Cleora...")
# Prepare Genre Interactions
# MovieLens
movielens_genres = movies_df.copy()
movielens_genres['genres'] = movielens_genres['genres'].str.split('|')
movielens_genres = movielens_genres.explode('genres')
movielens_genres = movielens_genres[movielens_genres['genres'] != '(no genres listed)']
movielens_genres['item_id'] = 'movielens_item_' + movielens_genres['movieId'].astype(str)
movielens_genres['genre_id'] = 'genre_' + movielens_genres['genres'].str.lower().str.replace(' ', '_').str.replace('-', '_')
ml_genre_interactions = movielens_genres[['item_id', 'genre_id']].rename(columns={'item_id': 'source', 'genre_id': 'target'})

# Steam
steam_genres = items_df_steam.copy()
steam_genres['genres'] = steam_genres['genres'].apply(lambda x: x if isinstance(x, list) else [])
steam_genres = steam_genres.explode('genres')
steam_genres['item_id'] = 'steam_item_' + steam_genres['app_id'].astype(str)
steam_genres['genre_id'] = 'genre_' + steam_genres['genres'].str.lower().str.replace(' ', '_').str.replace('-', '_')
st_genre_interactions = steam_genres[['item_id', 'genre_id']].rename(columns={'item_id': 'source', 'genre_id': 'target'})

# User-Item (from training split)
ui_interactions = train_interactions[['user_id', 'item_id']].rename(columns={'user_id': 'source', 'item_id': 'target'})

# Combine
cleora_interactions = pd.concat([ui_interactions, ml_genre_interactions, st_genre_interactions])

print("Training Cleora...")
cleora = CleoraGenerator(embedding_dim=128)
try:
    # We pass the combined dataframe. 
    # CleoraGenerator uses groupby(user_col)[item_col] and vice-versa.
    # We map 'source' -> 'user_col' and 'target' -> 'item_col'.
    cleora.train(cleora_interactions, user_col='source', item_col='target')
    cleora.save('models/cleora_model.pth') # Saving as pickle inside
    print("Cleora saved.")
except ImportError:
    print("Skipping Cleora (pycleora not installed).")